# <font color="#418FDE" size="6.5" uppercase>**D: ML Major Challenges**</font>
----

> Last update: 20231208

By the end of this lecture, you will be able to
* Explain key challenges in ML, such as unbalanced data, overfitting, & gradient-related issues, & demonstrate how to address them.


## **1. Unbalanced Data**

Unbalanced data is a significant challenge in ML, impacting the performance & reliability of predictive models. In scenarios where data is unbalanced, one class or category of data significantly outnumbers other classes, leading to a skewed distribution. This disproportion often results in models that are biased towards the majority class, compromising their ability to accurately predict or identify instances of the minority class. The issue is exacerbated when the minority class holds more significance, which is often the case in many real-world applications. The problem of unbalanced data is not just confined to a reduction in model accuracy but extends to impacting the model's generalizability & robustness, leading to potential misinterpretations & erroneous decisions based on the model's output.

Here are a few examples with respect to Mechanical Engineering:

> **Advanced Manufacturing**: In advanced manufacturing, unbalanced data can lead to predictive models that are inefficient in identifying defects or anomalies in production processes, as the instances of defects are typically much rarer compared to normal operation data.

> **Biomechanical Engineering**: In biomechanical engineering, data unbalance can affect models designed for predicting irregularities in human biomechanics or prosthetic alignment, where the majority of data might represent normal biomechanical patterns, overshadowing the critical minority data representing abnormalities.

> **Fluid Mechanics & Thermal Science**: In the realm of fluid mechanics & thermal science, unbalanced data could skew predictive models in predicting rare but critical phenomena like turbulent flows or thermal instabilities, as the data for such events are significantly less than for stable flow or thermal conditions.

> **Hypersonic Technologies**: For hypersonic technologies, where high-speed flight data is scarce compared to subsonic data, unbalanced datasets might lead to models that are not well-equipped to predict or analyze phenomena unique to hypersonic speeds.

> **Robotics, Dynamics, & Controls**: In robotics, dynamics, & controls, unbalanced data can impair the development of algorithms for rare but critical scenarios, such as emergency handling or unusual maneuvering, as these situations are less represented in datasets compared to standard operations.

> **Solids/Mechanics of Materials**: In the field of solids/mechanics of materials, unbalanced data can hinder the ability of models to predict rare material failure modes or stress responses, since most collected data might reflect normal material behavior under standard conditions.


Unbalanced datasets can lead to biased models that perform poorly on the minority class. In this section, we will see a few strategies to address this issue.

### **1.1. Resampling**

Balance the dataset by oversampling the minority class or undersampling the majority class.

In [ ]:
#@title Example of Resampling
'''
Runtime: CPU
make_classification: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
RandomOverSampler   https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.RandomOverSampler.html
'''
import numpy as np
from sklearn.datasets import make_classification
from imblearn.over_sampling import RandomOverSampler

# Generate synthetic imbalanced classification dataset
datain_tr, dataou_tr = make_classification(n_samples=1000, n_features=20, n_informative=2, n_redundant=0,
                                           n_clusters_per_class=1, weights=[0.95], flip_y=0,
                                           random_state=np.random.randint(1000))

print("Total Number of Data Points Before Resampling:  ", len(dataou_tr))
print("Number of Class 1 Data Points Before Resampling:", np.count_nonzero(dataou_tr))

# Resample the dataset
ros = RandomOverSampler(random_state=np.random.randint(1000))
datain_tr_res, dataou_tr_res = ros.fit_resample(datain_tr, dataou_tr)

print(" ")
print("Total Number of Data Points After Resampling:   ", len(dataou_tr_res))
print("Number of Class 1 Data Points After Resampling: ", np.count_nonzero(dataou_tr_res))

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **1.2. Weighted Classes**

Assign higher penalization weights to the minority class during training.

In [ ]:
#@title Example of Weighted Classes
'''
Runtime: CPU
make_classification:   https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
compute_class_weight: https://scikit-learn.org/stable/modules/generated/sklearn.utils.class_weight.compute_class_weight.html
'''
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.datasets import make_classification

# Generate synthetic imbalanced classification dataset
datain_tr, dataou_tr = make_classification(n_samples=1000, n_features=20, n_informative=2, n_redundant=0,
                                           n_clusters_per_class=1, weights=[0.95], flip_y=0,
                                           random_state=np.random.randint(1000))

# Compute class weights
class_weights = compute_class_weight('balanced', classes=np.unique(dataou_tr), y=dataou_tr)
class_weights_dict = {i0 : class_weights[i0] for i0 in range(len(class_weights))}

# Build a simple model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_dim=20),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Fit the model with class weights
model.fit(datain_tr, dataou_tr, epochs=5, class_weight=class_weights_dict)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **1.3. Synthetic Data Generation**

Use methods like <a href="https://towardsdatascience.com/smote-synthetic-data-augmentation-for-tabular-data-1ce28090debc"> Synthetic Minority Over-sampling Technique (SMOTE)</a> to generate synthetic examples of the minority class.

In [ ]:
#@title Example of Synthetic Data Generation
'''
Runtime: CPU
make_classification: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
SMOTE:              https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html
'''
# Import necessary libraries
import numpy as np
from sklearn.datasets import make_classification
from imblearn.over_sampling import SMOTE

# Generate synthetic data
datain_tr, dataou_tr = make_classification(n_samples=1000, n_features=20, n_informative=2, n_redundant=10,
                                           n_clusters_per_class=1, weights=[0.99], flip_y=0, random_state=np.random.randint(1000))

print("Total Number of Data Points Before SMOTE:  ", len(dataou_tr))
print("Number of Class 1 Data Points Before SMOTE:", np.count_nonzero(dataou_tr))

# Apply SMOTE to balance the data
smote = SMOTE(random_state=np.random.randint(1000))
datain_tr_res, dataou_tr_res = smote.fit_resample(datain_tr, dataou_tr)

print(" ")
print("Total Number of Data Points After SMOTE:   ", len(dataou_tr_res))
print("Number of Class 1 Data Points After SMOTE: ", np.count_nonzero(dataou_tr_res))

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **1.4. Anomaly Detection**

Treat the problem as anomaly detection if the minority class is rare.

In [ ]:
#@title Example of Anomaly Detection
'''
Runtime: CPU
make_classification: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
IsolationForest:    https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
'''
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import IsolationForest

# Generate synthetic imbalanced classification dataset
datain_tr, _ = make_classification(n_samples=1000, n_features=20, n_informative=2, n_redundant=0,
                                   n_clusters_per_class=1, weights=[0.95], flip_y=0,
                                   random_state=np.random.randint(1000))

# Fit Isolation Forest for anomaly detection
iso_forest = IsolationForest(random_state=np.random.randint(1000))
iso_forest.fit(datain_tr)

# Predict anomalies (outliers are labeled as -1)
outliers_pred = iso_forest.predict(datain_tr)

print(outliers_pred)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

## **2. Overfitting**

Overfitting is a prevalent challenge in ML, where a model learns not only the underlying patterns in the training data but also its noise & random fluctuations. This results in a model that performs exceptionally well on the training data but poorly on new, unseen data. Overfitting occurs when a model is excessively complex relative to the amount & noise of the input data, capturing spurious correlations instead of the actual underlying relationships. This leads to a lack of generalizability, meaning the model's predictions become less accurate & reliable when applied to real-world situations outside the training dataset. The phenomenon of overfitting undermines the essential objective of ML models to make accurate predictions or classifications on new, unseen data.

<div align="left">
  <img src="https://github.com/mhrafiei/figures/blob/main/overfitting.png?raw=true" width="50%">
  <br>
  <figcaption>Figure: Overfitted, Fitted, & Underfitted Models</figcaption>
</div>

Here are a few examples with respect to Mechanical Engineering:

> **Advanced Manufacturing**: In advanced manufacturing, overfitting can result in models that perfectly predict equipment behavior or defect detection on training data but fail in real-world scenarios, missing critical manufacturing anomalies or falsely identifying defects.

> **Biomechanical Engineering**: Overfitting in biomechanical engineering can cause models to precisely mimic the training biomechanical data but inaccurately represent or predict human movement patterns or prosthetic alignments in actual clinical conditions.

> **Fluid Mechanics & Thermal Science**: In fluid mechanics & thermal science, overfit models might precisely simulate laboratory conditions but inadequately predict fluid behavior or thermal responses in varied & complex real-world environments.

> **Hypersonic Technologies**: For hypersonic technologies, overfitting can lead to models that are highly tuned to specific test flight data but are incapable of generalizing to the broader range of conditions encountered in actual hypersonic flight.

> **Robotics, Dynamics, & Controls**: In robotics, dynamics, & controls, overfitting can create algorithms that work perfectly in controlled environments but fail in dynamic real-world scenarios, impacting the robot's adaptability & decision-making.

> **Solids/Mechanics of Materials**: In solids/mechanics of materials, overfitting can result in models that accurately predict material behavior under specific test conditions but fail to generalize to different stress states or environmental conditions, potentially leading to incorrect assessments of material durability or safety.


In this section, we will see a few strategies to address this issue.

### **2.1. Cross-Validation**

Cross-validation is a vital technique in training NNs, particularly for addressing overfitting, a common challenge given their complexity & capacity for learning detailed patterns. In NNs, overfitting occurs when the model learns not only the underlying patterns in the training data but also the noise & specific details that do not generalize to unseen data. Cross-validation helps mitigate this by dividing the dataset into multiple smaller sets or 'folds'. The most common form is $k$-fold cross-validation, where the data is split into $k$ subsets. The model is then trained $k$ times, each time using a different subset as the validation set & the remaining data as the training set. This process allows the model to be tested on various unseen data subsets, providing a more robust evaluation of its performance & generalizability. Cross-validation ensures that the model's effectiveness is not merely a result of peculiarities in a single split of training & validation data. It's especially crucial in scenarios with limited data, as it maximizes both the training & validation utility of available data. However, in the context of NNs, cross-validation can be computationally intensive due to the need to train multiple models, & careful consideration must be given to balance between comprehensive evaluation & computational feasibility. This technique, combined with proper model architecture selection & regularization, forms a comprehensive approach to combatting overfitting in NN training.

In [ ]:
#@title Example of Cross-Validation
import numpy as np
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.model_selection import KFold

'''
Runtime: CPU
make_classification: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
KFold             : https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html
'''

# Generate synthetic dataset
datain_tr, dataou_tr = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=np.random.randint(1000))

# Define KFold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=np.random.randint(1000))

# Model definition
def fun_create_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(20,)),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Perform KFold cross-validation
for i0, (index_tr, index_vl) in enumerate(kf.split(datain_tr)):
    print(50*'-')
    print("Fold # {:02d}".format(i0+1))
    print(50*'-')
    # Create data for this fold
    datain_tr_fold, datain_vl_fold = datain_tr[index_tr], datain_tr[index_vl]
    dataou_tr_fold, dataou_vl_fold = dataou_tr[index_tr], dataou_tr[index_vl]

    # Create & train model for this fold
    model = fun_create_model()
    model.fit(datain_tr_fold, dataou_tr_fold, epochs=5, validation_data=(datain_vl_fold, dataou_vl_fold))

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **2.2. Regularization**

Regularization is a fundamental technique in ML to prevent overfitting by penalizing model complexity. L1 & L2 regularization are crucial techniques in the context of NNs, where they help to prevent overfitting, a common issue given the high complexity & capacity of these models. In NNs, L1 regularization (Lasso) involves adding the absolute values of the weights as a penalty to the loss function. This can lead to sparse networks where some weights are zero, effectively simplifying the model by removing certain connections. On the other hand, L2 regularization (Ridge) adds the square of the weights as a penalty term. This doesn't lead to sparse networks like L1 but rather encourages the weights to be small, distributing the influence across various nodes & layers more evenly. Both these methods add a regularization term controlled by a hyperparameter, typically denoted as $\lambda$, which determines the strength of the penalty. The choice between L1 & L2 in NNs depends on the specific requirements & architecture of the model. L1 is often preferred when feature selection or network sparsity is desired, while L2 is commonly used for general weight decay, promoting smaller & more distributed weight values which can lead to more robust & generalizable models. It's noteworthy that applying these regularization techniques in deep learning frameworks requires careful tuning of the regularization parameter to balance between underfitting & overfitting while maintaining the model's capacity to learn complex patterns.

In [ ]:
#@title Example of Regularization
'''
Runtime: CPU
make_classification:                 https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
TensorFlow Regularization Overview: https://www.tensorflow.org/api_docs/python/tf/keras/regularizers
'''
import numpy as np
import tensorflow as tf
from sklearn.datasets import make_classification
from tensorflow.keras import regularizers

# Generate synthetic dataset
datain_tr, dataou_tr = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=np.random.randint(1000))

# Model definition with L1 & L2 regularization
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(20,), kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Fit the model
model.fit(datain_tr, dataou_tr, epochs=10, batch_size=32)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **2.3. Pruning**


Pruning in NNs & ML is a technique aimed at reducing model complexity to combat overfitting, which is a key challenge especially in deep learning models due to their large number of parameters. Overfitting occurs when a model learns not only the underlying patterns in the training data but also the noise, leading to poor generalization to new, unseen data. Pruning addresses this by systematically removing parts of the model that contribute the least to its predictive power. In NNs, this typically involves the elimination of weights, neurons, or even entire layers that have minimal impact on the output. The idea is to simplify the network by reducing the number of parameters, thereby making it less prone to memorizing the training data & more focused on learning generalizable patterns. Pruning can be done in various ways, such as removing weights below a certain threshold or using more sophisticated techniques that consider the overall impact of a weight on model performance. This process often results in a more compact, efficient, & generalizable model. Pruning not only helps in reducing overfitting but also in decreasing the computational load & memory usage, which is particularly beneficial in deploying models to resource-constrained environments like mobile devices. It's a balance, however, as excessive pruning can lead to underfitting, where the model becomes too simple to capture the underlying patterns in the data. Thus, careful tuning is required to find the optimal level of pruning that maintains model performance while reducing complexity ([Han et al. 2015](https://arxiv.org/abs/1506.02626)).

<div align="left">
  <img src="https://github.com/mhrafiei/figures/blob/main/pruningexample.png?raw=true" width="75%">
  <br>
  <figcaption>Figure: Example of NN Pruning (<a href=https://sp2023.ieee-security.org/downloads/SP23-posters/sp23-posters-paper35-final_version_2_page_abstract.pdf >Ghazvinian et al. 2023</a>)</figcaption>
</div>

Pruning is not directly supported in TensorFlow. However, the TensorFlow Model Optimization Toolkit offers sparsity-based pruning. Here's a simple example with scikit-learn's decision tree classifier.

In [ ]:
#@title Example of Pruning
'''
Runtime: CPU
make_classification:    https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
DecisionTreeClassifier: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
'''

import numpy as np
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier

# Generate synthetic dataset
datain_tr, dataou_tr = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=np.random.randint(1000))

# Decision tree with pruning (max_depth is a simple way to prune)
model = DecisionTreeClassifier(max_depth=3, random_state=np.random.randint(1000))
model.fit(datain_tr, dataou_tr)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **2.4. Early Stopping**


Early stopping is a widely used regularization technique in NNs & ML to prevent overfitting. It is based on the principle of monitoring the model's performance on a validation set during the training process & stopping the training once the performance begins to deteriorate or ceases to improve significantly. Overfitting occurs when a model learns not only the underlying patterns but also the noise in the training data, resulting in poor generalization to new, unseen data. Early stopping counters this by halting the training process before the model reaches this point. The procedure typically involves setting aside a portion of the training data as a validation set. During training, the model is evaluated on this validation set at regular intervals. If the model's performance on the validation set starts to degrade, or if it does not improve for a specified number of epochs (a condition often referred to as a "patience" parameter), the training is stopped. The model's state at the point of minimal validation loss (or accuracy) is often saved as the final model. Early stopping thus acts as a practical way to balance between underfitting & overfitting: it allows the model to learn sufficiently from the training data while ensuring it retains the ability to generalize well to new data. This technique is straightforward to implement & can be highly effective, making it a staple in training NN models.

In [ ]:
#@title Example of Early Stopping
'''
Runtime: CPU
make_classification: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
EarlyStopping:      https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
'''

import numpy as np
import tensorflow as tf
from sklearn.datasets import make_classification
from tensorflow.keras.callbacks import EarlyStopping

# Generate synthetic dataset
datain_tr, dataou_tr = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=np.random.randint(1000))
datain_vl, dataou_vl = make_classification(n_samples=200, n_features=20, n_classes=2, random_state=np.random.randint(1000))

# Model definition
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(20,)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_accuracy',
                               patience=5,
                               restore_best_weights=True) # 'val_loss'

# Fit the model with early stopping
model.fit(datain_tr, dataou_tr, epochs=50, validation_data=(datain_vl, dataou_vl), callbacks=[early_stopping])

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

## **3. Gradient-Related Issues**

Gradient-related issues, specifically vanishing & exploding gradients, are critical challenges in ML, particularly impacting the training & convergence of deep NN models. Vanishing gradients occur when the gradients of the loss function become increasingly small as they propagate back through the layers in a deep network, leading to minimal updates of the weights in the early layers. This issue hinders the network's ability to learn effectively, especially complex patterns, potentially resulting in underfitting. Conversely, exploding gradients happen when these gradients become excessively large during backpropagation, causing overly large updates to the network weights & often leading to model divergence & instability. Both problems can severely compromise the learning process, affecting the model's performance, convergence, & ability to generalize to new data.

Here are a few examples with respect to Mechanical Engineering:

> **Advanced Manufacturing**: In advanced manufacturing, gradient-related issues can lead to poorly trained models that fail to predict equipment failures or detect manufacturing defects, as the models might not converge effectively to learn from the complex, multi-layered data typical in manufacturing processes.

> **Biomechanical Engineering**: In biomechanical engineering, these gradient issues can result in models that are ineffective at predicting complex human movement patterns or diagnosing biomechanical disorders, as the models may not learn the deeper, subtle relationships in the biomechanical data due to poor convergence.

> **Fluid Mechanics & Thermal Science**: For fluid mechanics & thermal science, exploding or vanishing gradients can prevent models from accurately simulating fluid dynamics or thermal processes, particularly in simulations involving complex, multi-scale phenomena where deep learning models need to learn from various layers of data.

> **Hypersonic Technologies**: In hypersonic technologies, gradient problems can hinder the development of predictive models for high-speed aerodynamics, as these models require learning from highly complex & layered data, which can be challenging if the gradients do not propagate effectively through the network.

> **Robotics, Dynamics, & Controls**: In robotics, dynamics, & controls, gradient-related issues can lead to the failure of models in learning sophisticated control strategies or dynamics, particularly in scenarios involving complex, multi-dimensional data from sensors & actuators.

> **Solids/Mechanics of Materials**: In solids & mechanics of materials, these issues can prevent models from accurately predicting material behavior under stress, as the intricate relationships in material properties & stress responses may not be effectively learned due to poor gradient propagation in the network.

In this section, we will see a few strategies to address this issue.

### **3.1. Gradient Clipping**

Gradient clipping is a technique used in training NNs to address the issue of exploding gradients, one of the common "Gradient-Related" issues. Exploding gradients occur when the gradients calculated during the backpropagation process become excessively large, causing the model's weights to update in large, unstable steps. This can lead to a divergent training process, where the model fails to converge & exhibits erratic behavior.

Gradient clipping mitigates this problem by setting a threshold value & then scaling down the gradients if they exceed this threshold. Essentially, it involves capping the gradients at a specified maximum value during training. This ensures that the gradients do not grow too large, maintaining stable training & preventing the weights from becoming too large. There are different ways to implement gradient clipping, but the most common approach is to rescale the gradients when their norm exceeds a certain threshold.

By employing gradient clipping, NN training becomes more stable, especially in deep learning models with many layers, where the risk of exploding gradients is higher. This technique is particularly useful in scenarios involving recurrent NNs (RNNs) & long short-term memory networks (LSTMs), which are more prone to the issue of exploding gradients due to their sequential data processing & extended training sequences. Gradient clipping helps in maintaining the model's ability to learn effectively over time without the training process becoming unstable or diverging, thereby aiding in the overall convergence & performance of the model.

In [ ]:
# @title
#@title Example of Gradient Clipping
'''
Runtime: CPU
make_classification:  https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
Read about clipnorm: https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam
'''

import numpy as np
import tensorflow as tf
from sklearn.datasets import make_classification

# Generate synthetic classification dataset
datain_tr, dataou_tr = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=np.random.randint(1000))

# Convert to TensorFlow tensors
datain_tr = tf.convert_to_tensor(datain_tr, dtype=tf.float32)
dataou_tr = tf.convert_to_tensor(dataou_tr, dtype=tf.float32)

# Model definition
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(20,)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile model with gradient clipping in the optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Fit the model
model.fit(datain_tr, dataou_tr, batch_size=32, epochs=10)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **3.2. Normalization Techniques**

Normalization techniques, particularly the use of Batch Normalization (BatchNorm) layers, play a pivotal role in addressing gradient-related issues in NNs & enhancing the overall training process. Batch Normalization, introduced by [Ioffe & Szegedy (2015)](https://arxiv.org/abs/1502.03167), is a technique to normalize the inputs of each layer within a network. By doing this, it helps in mitigating the problem of internal covariate shift, where the distribution of each layer’s inputs changes during training, as the parameters of the previous layers change. This stabilization of input distributions leads to more stable & faster convergence of the network. BatchNorm works by normalizing the output of a previous activation layer by subtracting the batch mean & dividing by the batch standard deviation. Consequently, this process helps in controlling the range of values through the network, which in turn addresses the issue of vanishing & exploding gradients. Gradients in deeper layers of the network are less likely to vanish or explode, as BatchNorm ensures that the values throughout the network maintain a consistent scale. Moreover, BatchNorm also allows the use of higher learning rates, which can further speed up the training of NNs. This normalization technique has become a standard in many deep learning architectures due to its effectiveness in stabilizing training, improving convergence rates, & generally leading to better overall model performance.

In [ ]:
# @title
#@title Example of Normalization Techniques
'''
Runtime: CPU
make_classification:            https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
TensorFlow BatchNormalization: https://www.tensorflow.org/api_docs/python/tf/keras/layers/BatchNormalization
'''

import numpy as np
import tensorflow as tf
from sklearn.datasets import make_classification

# Generate synthetic classification dataset
datain_tr, dataou_tr = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=np.random.randint(1000))

# Convert to TensorFlow tensors
datain_tr = tf.convert_to_tensor(datain_tr, dtype=tf.float32)
dataou_tr = tf.convert_to_tensor(dataou_tr, dtype=tf.float32)

# Model definition with Batch Normalization
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(64, input_shape=(20,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Fit the model
model.fit(datain_tr, dataou_tr, batch_size=32, epochs=10)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **3.3. Choosing Proper Activation Functions**

Choosing proper activation functions is a critical aspect in designing NNs to mitigate gradient-related issues like vanishing & exploding gradients. Activation functions introduce non-linearities into the network, allowing it to learn complex patterns in the data. Traditional functions like the sigmoid or tanh can lead to vanishing gradients, especially in deep networks, as their derivatives are small & diminish rapidly as the magnitude of the input grows large or small. This results in gradients that are too small to make significant updates in the earlier layers of the network. To address this, the Rectified Linear Unit (ReLU) & its variants (like Leaky ReLU, Parametric ReLU) have become popular. ReLU, defined as $f(x)=\max(0,x)$, has the advantage of a constant gradient for positive inputs, which greatly alleviates the vanishing gradient problem & helps in faster convergence. However, ReLU & its variants are not without issues; for example, ReLU can lead to dead neurons where some neurons can become inactive & stop contributing to the learning process. This has led to the development of more sophisticated functions like ELU (Exponential Linear Unit) or Swish, which are designed to maintain robust gradients & reduce the likelihood of dead neurons.

LeakyReLU is a variant of the standard ReLU function & is designed to address one of its primary drawbacks: the dying ReLU problem, where neurons can become inactive & stop learning entirely. In contrast to ReLU, which outputs zero for all negative inputs, LeakyReLU allows a small, non-zero gradient when the unit is inactive (i.e., for negative input values). It is defined as $f(x)=x$ if $x>0$ & $f(x)=\alpha x$ if $x≤0$, where $\alpha$ is a small constant. This slight slope for negative values ensures that there is always a gradient flowing through the neuron, reducing the likelihood of the gradients vanishing during backpropagation. This makes LeakyReLU particularly useful in deeper networks, where vanishing gradients are more prevalent. The parameter $\alpha$ is typically a small value such as 0.01. By maintaining a pathway for backpropagated errors to flow, even for negative inputs, LeakyReLU helps in maintaining active neuron states throughout the network, contributing to more consistent learning & reducing the risk of gradient issues.

The choice of activation function can significantly impact the training dynamics & overall performance of an NN, making it a key consideration in addressing gradient-related challenges.

In [ ]:
# @title
#@title Example of Choosing Proper Activation Functions
'''
Runtime: CPU
make_classification:     https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
TensorFlow Activations: https://www.tensorflow.org/api_docs/python/tf/keras/activations
'''

import numpy as np
import tensorflow as tf
from sklearn.datasets import make_classification

# Set random seed for reproducibility
random_state = np.random.randint(1000)

# Generate synthetic classification dataset
datain_tr, dataou_tr = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=random_state)

# Model definition using LeakyReLU activation function
model = tf.keras.models.Sequential([
    # Adding the LeakyReLU as an advanced activation function
    tf.keras.layers.Dense(64, input_shape=(20,), kernel_initializer='he_normal'),
    tf.keras.layers.LeakyReLU(alpha=0.01),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Convert data to tf.float32
datain_tr = tf.convert_to_tensor(datain_tr, dtype=tf.float32)
dataou_tr = tf.convert_to_tensor(dataou_tr, dtype=tf.float32)

# Fit the model
model.fit(datain_tr, dataou_tr, batch_size=32, epochs=10)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **3.4. Careful Initialization**

Careful initialization of weights in NNs is a crucial step to mitigate gradient-related issues such as vanishing & exploding gradients. Proper initialization sets the stage for an effective training process by ensuring that the gradients are neither too large nor too small at the start, which can significantly impact the convergence of the network. Traditional methods like initializing weights with small random values or zeros can lead to problems in deep networks; for example, zero initialization can result in a lack of symmetry breaking, while small random values can exacerbate the vanishing or exploding gradient problem. To address this, more sophisticated initialization techniques have been developed. One prominent approach is the Xavier/Glorot initialization, which considers the size of the previous & next layer in the network to normalize the weights ([more](https://towardsdatascience.com/xavier-glorot-initialization-in-neural-networks-math-proof-4682bf5c6ec3)). Another approach is the He (Kaiming) initialization ([He et al. 2015](https://arxiv.org/abs/1502.01852v1)), which is particularly effective for networks using ReLU activation functions. This method sets the initial weights by considering the size of the previous layer, leading to an initialization that maintains the variance of the activations & gradients across layers. Such careful, methodical initialization ensures that the signal does not vanish or blow up as it passes through each layer, enabling deeper networks to be trained more effectively & efficiently. By laying a solid foundation for the training process, careful initialization of weights is instrumental in addressing & preventing gradient-related challenges in NNs.

In [ ]:
# @title
#@title Example of Careful Initialization
'''
Runtime: CPU
make_classification:             https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
TensorFlow Kernel Initializers: https://www.tensorflow.org/api_docs/python/tf/keras/initializers
'''

import numpy as np
import tensorflow as tf
from sklearn.datasets import make_classification

# Generate synthetic classification dataset
datain_tr, dataou_tr = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=np.random.randint(1000))

# Convert to TensorFlow tensors
datain_tr = tf.convert_to_tensor(datain_tr, dtype=tf.float32)
dataou_tr = tf.convert_to_tensor(dataou_tr, dtype=tf.float32)

# Model definition with He Initialization
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(64, activation='relu', kernel_initializer='he_normal', input_shape=(20,)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Fit the model
model.fit(datain_tr, dataou_tr, batch_size=32, epochs=10)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

## **4. Model Complexity**

Model complexity in ML refers to the intricacy of the model's structure, including the number of parameters & layers. While complex models have the potential to capture & learn from subtle patterns in large & intricate datasets, they also come with significant challenges. One primary issue is the risk of overfitting, where a model learns the noise & specificities of the training data so well that it fails to generalize to new, unseen data. Complex models also require more computational resources & longer training times, making them less efficient & more challenging to implement & use in real-world applications. Additionally, they are often less interpretable, making it difficult to understand the decision-making process of the model, which is crucial in many applications.

Here are a few examples with respect to Mechanical Engineering:

> **Advanced Manufacturing**: In advanced manufacturing, overly complex models might overfit to specific manufacturing conditions or noise in the data, leading to inaccurate predictions or failure to generalize across different manufacturing setups or processes.

> **Biomechanical Engineering**: In biomechanical engineering, complex models might capture noise in biomechanical data, leading to inaccurate predictions or classifications of biomechanical patterns, which could be detrimental in applications like prosthetic design or rehabilitation.

> **Fluid Mechanics & Thermal Science**: For fluid mechanics & thermal science, highly complex models could fail to generalize to different fluid dynamics or thermal conditions, making them less reliable for predicting behavior in varied real-world scenarios.

> **Hypersonic Technologies**: In hypersonic technologies, the use of overly complex models could lead to poor generalization beyond the specific conditions & data on which they were trained, hindering their applicability in varied hypersonic scenarios.

> **Robotics, Dynamics, & Controls**: Complex models in robotics might overfit to training scenarios, failing to adapt or respond correctly to unanticipated situations or environments, which is crucial for robotic systems' adaptability & safety.

> **Solids/Mechanics of Materials**: In solids & mechanics of materials, overly complex models could lead to misinterpretation of material behavior, especially under stress or in unusual conditions, due to overfitting to specific datasets.

Complex models may require significant computational resources & time to train. In this section, we will see a few strategies to address this issue.

### **4.1. Simplification**

Simplification in NNs & ML, as a response to model complexity issues, involves the strategic use of less complex, more streamlined models. This approach addresses challenges such as overfitting, high computational costs, & difficulties in model interpretation that often accompany highly complex models like deep NNs. By adopting simpler models, such as linear regressions or smaller NNs, practitioners can achieve faster training times, easier model tuning, & enhanced interpretability. This simplicity can also lead to better generalization on unseen data, as complex models can become excessively tailored to the training dataset, capturing noise rather than underlying patterns. Moreover, simpler models are more manageable & can be more effective in scenarios where data is limited or the underlying relationships in the data are not overly complex. Ultimately, model simplification aligns with the principle of [Occam's Razor](https://math.ucr.edu/home/baez/physics/General/occam.html) in scientific modeling: the simplest solution is often preferable, assuming it adequately captures the phenomenon being modeled.

In [ ]:
# @title
#@title Example of Simplification
'''
Runtime: CPU
Look into https://keras.io/api/applications/
'''

### **4.2. Dimensionality Reduction**

Dimensionality reduction is a critical technique in NNs & ML to address "Model Complexity" issues. In many real-world scenarios, datasets come with a high number of features (high dimensionality), which can significantly increase the complexity of a ML model. This complexity not only leads to increased computational costs & training times but also exacerbates the risk of overfitting, as models with too many parameters tend to learn noise & spurious correlations in the data. Dimensionality reduction techniques aim to reduce the number of input variables to a manageable size, while retaining as much of the meaningful information as possible. Techniques like [Principal Component Analysis (PCA)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html), [t-Distributed Stochastic Neighbor Embedding (t-SNE)](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html), & [autoencoders](https://scikit-learn.org/stable/modules/neural_networks_unsupervised.html) are commonly used for this purpose. By transforming the original high-dimensional data into a lower-dimensional space, these methods help in simplifying the model, making it more computationally efficient & less prone to overfitting. In NNs, dimensionality reduction can be implicitly performed through layers that consolidate information (like convolutional layers in CNNs). Effective dimensionality reduction leads to simpler, more interpretable models that are better suited for generalization, thus addressing the challenges posed by model complexity.

In [ ]:
# @title
#@title Example of Dimensionality Reduction
'''
Runtime: CPU
make_classification:  https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
PCA:                 https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
'''

import numpy as np
from sklearn.decomposition import PCA
from sklearn.datasets import make_classification
import tensorflow as tf

# Generate synthetic dataset
random_state = np.random.randint(1000)
datain_tr, dataou_tr = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=random_state)

# Apply PCA for dimensionality reduction
pca = PCA(n_components=10, random_state=random_state)
datain_tr_reduced = pca.fit_transform(datain_tr)

# Convert reduced data to TensorFlow tensors
datain_tr_reduced = tf.convert_to_tensor(datain_tr_reduced, dtype=tf.float32)
dataou_tr = tf.convert_to_tensor(dataou_tr, dtype=tf.float32)

# Train a model on the reduced data
model = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='relu', input_shape=(10,)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Fit the model
model.fit(datain_tr_reduced, dataou_tr, epochs=10, batch_size=32)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **4.3. Transfer Learning**

Transfer learning is a powerful approach in NNs & ML that addresses "Model Complexity" issues, especially in scenarios where collecting a large, comprehensive dataset is challenging or when computational resources are limited. In transfer learning, a model developed for a specific task is reused as the starting point for a model on a second task. This is particularly prevalent in deep learning, where [large NNs pre-trained on massive datasets](https://keras.io/api/applications/) (like ImageNet for image tasks) are adapted to new tasks with relatively less data. The key advantage here is that these pre-trained models have already learned a set of features that are potentially useful across a wide range of tasks, thus bypassing the need for training a complex model from scratch. This approach not only saves significant time & computational resources but also helps in mitigating overfitting, as the model is not trained exclusively on a small dataset. Instead, it leverages patterns learned from a larger dataset, enhancing generalization. Transfer learning is particularly beneficial in tasks where the data is limited or where the complexity of training a large model from scratch is not feasible. By fine-tuning the pre-trained model - adjusting the final layers & parameters to the specific task at hand - transfer learning achieves a balance, harnessing the power of complex models while reducing the risks associated with model complexity.

In [ ]:
# @title
#@title Example of Transfer Learning
'''
Runtime: GPU $$$
TensorFlow Applications: https://www.tensorflow.org/api_docs/python/tf/keras/applications
TensorFlow Datasets:     https://www.tensorflow.org/api_docs/python/tf/keras/datasets
CIFAR10:                 https://www.cs.toronto.edu/~kriz/cifar.html
'''

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.utils import to_categorical

# Load CIFAR-10 data
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = cifar10.load_data()

# Preprocess the data
datain_tr = datain_tr.astype('float32') / 255
datain_vl = datain_vl.astype('float32') / 255
dataou_tr = to_categorical(dataou_tr, 10)
dataou_vl = to_categorical(dataou_vl, 10)

# Upscale images from 32x32 to 96x96
datain_tr = tf.image.resize(datain_tr, [96, 96])
datain_vl = tf.image.resize(datain_vl, [96, 96])

'''''''''''''''''''''''''''''''''''''''''''''
Without Transfer Learning
'''''''''''''''''''''''''''''''''''''''''''''

# Load MobileNetV2 without any pre-trained weights (weights are initiated randomly)
model_base = MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights=None)

# Add custom layers
x = model_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(10, activation='softmax')(x)

# Define the model
model = Model(inputs=model_base.input, outputs=predictions)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Fit the model for 2 epochs
print(50*'-')
print("Without Transfer Learning ")
print(50*'-')
model.fit(datain_tr, dataou_tr, batch_size=32, epochs=2, validation_data=(datain_vl, dataou_vl))

'''''''''''''''''''''''''''''''''''''''''''''
With Transfer Learning
'''''''''''''''''''''''''''''''''''''''''''''

# Load MobileNetV2 with pre-trained ImageNet weights
model_base = MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights='imagenet')

# Freeze the layers of the base model
for layer in model_base.layers:
    layer.trainable = False

# Add custom layers
x = model_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(10, activation='softmax')(x)

# Define the model
model = Model(inputs=model_base.input, outputs=predictions)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Fit the model for 2 epochs
print(50*'-')
print("With Transfer Learning ")
print(50*'-')
model.fit(datain_tr, dataou_tr, batch_size=32, epochs=2, validation_data=(datain_vl, dataou_vl))

# Summary of model
model.summary()

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

## **5. Poor Generalization**



Poor generalization in ML occurs when a model, trained on a specific set of data, fails to perform adequately on new, unseen data. This problem often arises from overfitting, where the model learns the noise & specific patterns in the training data to such an extent that it impairs its ability to generalize. Models with poor generalization are limited in their practical applicability, as their predictions or classifications become unreliable when exposed to real-world data that slightly deviates from the training set. This challenge is particularly acute in complex models with many parameters, as they have a higher capacity to learn detailed, non-generalizable features of the training data. The core issue is that while these models achieve high accuracy on their training data, their performance degrades significantly on new data, making them less effective for real-world applications.

Here are a few examples with respect to Mechanical Engineering:

> **Advanced Manufacturing**: Poor generalization in advanced manufacturing could lead to ML models that are highly accurate in detecting defects or optimizing processes under specific conditions but fail to maintain accuracy when applied to different manufacturing settings or new types of materials.

> **Biomechanical Engineering**: In biomechanical engineering, poor generalization might result in models that are effective in analyzing movement or diagnosing conditions in a controlled dataset but are unable to accurately assess different populations or varied biomechanical conditions.

> **Fluid Mechanics & Thermal Science**: Models with poor generalization in fluid mechanics & thermal science may perform well in simulating specific fluid dynamics or thermal processes but could fail in predicting behavior under different physical conditions or in different fluid systems.

> **Hypersonic Technologies**: For hypersonic technologies, poor generalization can be a major drawback, leading to models that are precise in specific test scenarios but unreliable in predicting the complexities of varied hypersonic environments.

> **Robotics, Dynamics, & Controls**: In robotics, dynamics, & controls, poor generalization can result in control algorithms that work efficiently in simulation or specific scenarios but fail in real-world environments or under varying operational conditions.

> **Solids/Mechanics of Materials**: In the field of solids & mechanics of materials, models that do not generalize well could lead to inaccurate predictions of material behavior under different stress conditions, impacting the design & safety of engineering structures.

In this section, we will see a few strategies to address this issue.

### **5.1. Data Augmentation**

Data augmentation is a widely used technique in NNs & ML to tackle the issue of poor generalization. This method involves artificially expanding the training dataset by creating modified versions of the existing data. In the context of image processing, this could mean applying various transformations like rotation, scaling, flipping, or altering brightness & contrast to generate new training samples. For text data, it might involve techniques like synonym replacement, sentence shuffling, or translation back-and-forth between languages. The fundamental idea is to simulate a broader range of scenarios & variations that the model might encounter in the real world, thus enhancing its ability to generalize. By training on this augmented dataset, the model is less likely to overfit to the noise & specific patterns of the original data & more likely to learn the underlying, generalizable features. Data augmentation effectively increases the diversity & quantity of training data without the need for additional data collection, which can be especially beneficial in scenarios where data is scarce or expensive to acquire. It helps in building more robust models that perform better on new, unseen data, addressing the critical challenge of poor generalization in ML.


In [ ]:
# @title
#@title Example of Data Augmentation
'''
Runtime: GPU $$$
ImageDataGenerator:   https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/image/ImageDataGenerator
TensorFlow Datasets:  https://www.tensorflow.org/api_docs/python/tf/keras/datasets
CIFAR10:              https://www.cs.toronto.edu/~kriz/cifar.html
'''

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load CIFAR-10 data
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = cifar10.load_data()

# Normalize pixel values
datain_tr, datain_vl = datain_tr / 255.0, datain_vl / 255.0

# Convert class vectors to binary class matrices (one-hot encoding)
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

# Data augmentation generator
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Build a simple CNN model
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model with data augmentation
model.fit(datagen.flow(datain_tr, dataou_tr, batch_size=32),
          epochs=3,
          validation_data=(datain_vl, dataou_vl),
          steps_per_epoch=len(datain_tr) // 32)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **5.2. Ensemble Methods**

[Ensemble methods](https://towardsdatascience.com/ensemble-methods-bagging-boosting-and-stacking-c9214a10a205) in NNs & ML are a powerful strategy to address the issue of poor generalization. These methods involve combining the predictions from multiple models to improve the overall performance & robustness of the system. The core principle behind ensemble methods is that a group of models, working together, can often make more accurate predictions than a single model. Common ensemble techniques include bagging, boosting, & stacking. Bagging (Bootstrap Aggregating) involves training multiple models independently on different subsets of the data & then averaging their predictions, as seen in Random Forests. Boosting, on the other hand, sequentially trains models, each trying to correct the errors of the previous ones, exemplified by algorithms like [Gradient Boosting Machines (GBM)](https://www.frontiersin.org/articles/10.3389/fnbot.2013.00021/full) & [XGBoost](https://machinelearningmastery.com/extreme-gradient-boosting-ensemble-in-python/). Stacking combines different types of models & uses their outputs as input to a final model to make predictions. By leveraging the strengths of various models & mitigating their individual weaknesses, ensemble methods reduce the risk of overfitting, leading to better generalization on unseen data. This collective decision-making approach ensures that the idiosyncrasies of a single model do not dominate the final prediction, making ensemble methods particularly effective in complex problems where poor generalization is a significant concern.

In [ ]:
# @title
#@title Example of Ensemble Methods
'''
Runtime: GPU $$$
TensorFlow Datasets:  https://www.tensorflow.org/api_docs/python/tf/keras/datasets
CIFAR10:              https://www.cs.toronto.edu/~kriz/cifar.html
Accuracy Score:       https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html
'''
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, MaxPooling2D, Dropout
from sklearn.metrics import accuracy_score

# Load CIFAR-10 data
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = cifar10.load_data()

# Normalize pixel values
datain_tr, datain_vl = datain_tr / 255.0, datain_vl / 255.0

# Convert class vectors to binary class matrices
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

# Define two different models
def fun_create_model_1():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(10, activation='softmax')
    ])
    return model

def fun_create_model_2():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
        MaxPooling2D(2, 2),
        Dropout(0.2),
        Conv2D(64, (3, 3), activation='relu'),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(10, activation='softmax')
    ])
    return model

model1 = fun_create_model_1()
model2 = fun_create_model_2()

# Compile both models
model1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train both models
print(50*'-')
print("Model 1 Training")
print(50*'-')
model1.fit(datain_tr, dataou_tr, epochs=1, validation_data=(datain_vl, dataou_vl), verbose=1)

print(50*'-')
print("Model 2 Training")
print(50*'-')
model2.fit(datain_tr, dataou_tr, epochs=1, validation_data=(datain_vl, dataou_vl), verbose=1)

# Predictions from both models
dataes_vl1 = model1.predict(datain_vl, verbose = 3)
dataes_vl2 = model2.predict(datain_vl, verbose = 3)

# Average ensemble predictions
ensemble_preds = (dataes_vl1 + dataes_vl2) / 2

# Evaluate the ensemble
ensemble_accuracy = accuracy_score(dataou_vl.argmax(axis=1), ensemble_preds.argmax(axis=1))
print(50*'-')
print(f"Ensemble Accuracy: {ensemble_accuracy}")
print(50*'-')

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **5.3. Hyperparameter Tuning**

Hyperparameter tuning is a critical process in NNs & ML that can significantly impact a model's generalization capabilities. Hyperparameters are the parameters whose values are set before the learning process begins, unlike the model parameters that are learned during training. These include learning rate, number of layers & neurons in an NN, batch size, & regularization parameters, among others. Proper hyperparameter tuning is essential to strike the right balance between underfitting & overfitting. If hyperparameters are not optimally set, a model might overfit to the training data, capturing noise & specific patterns that do not generalize to unseen data. Conversely, overly conservative hyperparameter settings can lead to underfitting, where the model fails to capture even the basic patterns in the data. Techniques like grid search, random search, or more advanced methods like [Bayesian optimization](https://www.run.ai/guides/hyperparameter-tuning/bayesian-hyperparameter-optimization#:~:text=Bayesian%20optimization%E2%80%94tuning%20hyperparameters%20using,of%20test%20set%20generalization%20tasks.) are used to systematically explore the hyperparameter space & find the optimal settings. This tuning process is often iterative & requires careful consideration of the trade-offs between model complexity & performance. By optimizing hyperparameters, one can improve a model's ability to learn generalized patterns from the training data, enhancing its performance on new, unseen data & thereby addressing the issue of poor generalization.


In [ ]:
# @title
#@title Install Keras Tuner
!pip install keras-tuner

In [ ]:
# @title
#@title Example of Hyperparameter Tuning
'''
Runtime: GPU $$$
TensorFlow Datasets:  https://www.tensorflow.org/api_docs/python/tf/keras/datasets
CIFAR10:              https://www.cs.toronto.edu/~kriz/cifar.html
Keras Tuner:          https://keras.io/keras_tuner/
RandomSearch:         https://keras.io/api/keras_tuner/tuners/random/
'''

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, MaxPooling2D
from kerastuner.tuners import RandomSearch

# Load CIFAR-10 data
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = cifar10.load_data()

# Normalize pixel values
datain_tr, datain_vl = datain_tr / 255.0, datain_vl / 255.0

# Convert class vectors to binary class matrices
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

# Model builder function for Keras Tuner
def fun_create_model(hp):
    model = Sequential()
    model.add(Conv2D(hp.Int('conv_units', min_value=32, max_value=256, step=32), (3, 3), activation='relu', input_shape=(32, 32, 3)))
    model.add(MaxPooling2D(2, 2))
    model.add(Flatten())
    model.add(Dense(hp.Int('dense_units', min_value=32, max_value=256, step=32), activation='relu'))
    model.add(Dense(10, activation='softmax'))
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Create a tuner
tuner = RandomSearch(
    fun_create_model,
    objective='val_accuracy',
    max_trials=5,  # Number of trials to run
    executions_per_trial=3,  # Number of models to train for each trial
    directory='my_dir',
    project_name='cifar10'
)

# Perform hyperparameter tuning
tuner.search(datain_tr, dataou_tr, epochs=2, validation_data=(datain_vl, dataou_vl))

# Get the optimal hyperparameters
hps_best = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best Hyperparameters: {hps_best.values}")

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

## **6. Scalability**

Scalability in ML refers to the ability of a model or algorithm to maintain or improve its performance when the size of the dataset or the complexity of the problem increases. This challenge becomes particularly prominent in real-world applications where data volumes are large & continually growing. A model that works well on a small dataset might struggle with larger datasets due to increased computational demands, memory constraints, or longer training times. Furthermore, as the complexity of the data increases, models may require more sophisticated features or architectures, which can also pose scalability challenges. Scalability issues can lead to reduced model accuracy, increased processing times, & difficulties in adapting models to handle larger or more complex datasets, limiting their practical applicability in dynamic & data-intensive environments.

Here are a few examples with respect to Mechanical Engineering:

> **Advanced Manufacturing**: In advanced manufacturing, scalability issues can hinder the deployment of ML models for real-time monitoring & control of manufacturing processes, as these models might struggle to process & analyze large-scale data generated by advanced manufacturing systems efficiently.

> **Biomechanical Engineering**: Scalability problems in biomechanical engineering could affect the ability of models to handle large-scale biomechanical data, such as data from widespread wearable sensor deployments, limiting their effectiveness in large-scale studies or personalized healthcare applications.

> **Fluid Mechanics & Thermal Science**: For fluid mechanics & thermal science, scalability issues can impede the development of models capable of simulating complex fluid dynamics or thermal systems, particularly when dealing with large-scale simulations or high-resolution data.

> **Hypersonic Technologies**: In hypersonic technologies, the challenge of scalability might limit the ability of models to process & analyze extensive datasets required for the design & testing of hypersonic vehicles, affecting the development & optimization of these technologies.

> **Robotics, Dynamics, & Controls**: Scalability issues in robotics can lead to difficulties in implementing ML models that need to process & react to vast amounts of sensory & operational data in real-time, impacting the efficiency & adaptability of robotic systems.

> **Solids/Mechanics of Materials**: In the field of solids & mechanics of materials, scalability challenges can hinder the ability of models to analyze large datasets from material testing or simulations, affecting the development of new materials & the understanding of material behaviors under various conditions.

In this section, we will see a few strategies to address this issue.

### **6.1. Distributed Computing**

Distributed computing is a key strategy in addressing scalability issues in NNs & ML. As models become more complex & datasets larger, the computational demands often exceed the capacity of a single machine or processor. Distributed computing tackles this challenge by spreading the computational tasks across multiple machines or processors, effectively parallelizing the workload. This approach is particularly vital for training large NNs or processing vast datasets, tasks that are computationally intensive & time-consuming. By distributing the workload, models can be trained faster, & larger datasets can be processed more efficiently, enhancing the scalability of ML algorithms. Techniques like data parallelism, where the dataset is split across different machines, & model parallelism, where different parts of an NN model are trained on different machines, are common methods of distributed computing in this field. Additionally, the use of GPU clusters & cloud-based computing resources further enhances the capability to handle large-scale ML tasks. Distributed computing not only addresses the issue of processing large volumes of data but also enables the application of more sophisticated & complex models, thereby overcoming the significant hurdle of scalability in ML.

In [ ]:
# @title
#@title Example of Distributed Computing
'''
Runtime: GPU $$$
TensorFlow Datasets:   https://www.tensorflow.org/api_docs/python/tf/keras/datasets
CIFAR10:               https://www.cs.toronto.edu/~kriz/cifar.htm
TensorFlow Distribute: https://www.tensorflow.org/api_docs/python/tf/distribute
MirroredStrategy:      https://www.tensorflow.org/api_docs/python/tf/distribute/MirroredStrategy
'''

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.utils import to_categorical

# Load CIFAR-10 data
(datain_tr, dataou_tr), _ = cifar10.load_data()
datain_tr = datain_tr.astype('float32') / 255
dataou_tr = to_categorical(dataou_tr, 10)

# Define a simple CNN model
def fun_create_model():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Distributed training using MirroredStrategy
strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    model = fun_create_model()

# Fit the model
model.fit(datain_tr, dataou_tr, epochs=5, batch_size=64)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **6.2. Online Learning**

Online learning, also known as [incremental learning](https://medium.com/analytics-vidhya/incremental-online-learning-9868861db880), is an approach in NNs & ML that addresses scalability issues, especially in the context of handling large-scale data or data streams. Unlike traditional batch learning, where a model is trained on the entire dataset at once, online learning involves continuously updating the model incrementally as new data arrives. This method is particularly advantageous when dealing with large datasets that are too big to fit into memory at once or when the data is generated in a continuous stream, such as in real-time applications. Online learning allows the model to adapt to new patterns in the data without the need to retrain from scratch with each new batch of data. This approach significantly reduces memory requirements & computational costs, as only a portion of the data needs to be processed at a time. Furthermore, online learning enables models to stay updated with the most recent data, making them more adaptable to changes & new trends. This continuous & incremental training approach is key for scalable ML solutions in dynamic environments where data is constantly evolving, ensuring that models remain effective & relevant over time.

In [ ]:
# @title
#@title Example of Online Learning
'''
Runtime: CPU
make_classification: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
'''
import numpy as np
import tensorflow as tf
from sklearn.datasets import make_classification

# Generate synthetic dataset
random_state = np.random.randint(1000)
datain_tr, dataou_tr = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=random_state)

# Convert to TensorFlow tensors
datain_tr = tf.convert_to_tensor(datain_tr, dtype=tf.float32)
dataou_tr = tf.convert_to_tensor(dataou_tr, dtype=tf.float32)

# Define a simple model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(20,)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Online learning simulation: Update model with small data batches sequentially
for i in range(0, len(datain_tr), 50):
    model.fit(datain_tr[i:i+50], dataou_tr[i:i+50], epochs=1, verbose=1)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **6.3. Data Sampling**

Data sampling, particularly the use of representative samples, is a crucial strategy in addressing scalability issues in NNs & ML. As datasets grow in size, processing the entire dataset can become computationally prohibitive, especially for complex models. Data sampling involves selecting a subset of the data that is representative of the whole dataset. This approach allows for a significant reduction in the size of the data to be processed, leading to reduced computational load & faster training times. The key is to ensure that the sampled data captures the essential characteristics & distributions of the full dataset, thereby maintaining the quality & reliability of the model's training process. Techniques like [stratified sampling](https://medium.com/analytics-vidhya/stratified-sampling-in-machine-learning-f5112b5b9cfe), where samples are taken from different subgroups or strata within the dataset, can help in maintaining this representativeness. By training models on these smaller, yet representative, samples, scalability issues related to large data volumes can be mitigated, making the application of complex ML models more feasible & efficient. This approach not only speeds up the model training but also enables handling larger datasets more effectively, ensuring that the models remain scalable & adaptable to large-scale & diverse data environments.

For data sampling & pipeline development, the TensorFlow Dataset API provides several benefits for data processing & input pipeline optimization in TensorFlow, making it an essential tool for efficient & scalable ML workflows. Here are some of its key benefits:

* Efficiency: The Dataset API allows for more efficient data loading & preprocessing. It enables parallel processing, where data can be preprocessed & loaded in parallel to the training process. This reduces the time the model spends waiting for data, leading to faster training.

* Scalability: It handles large datasets effectively, even those too large to fit into memory. By enabling data to be streamed from disk & processed incrementally, the Dataset API ensures scalability & efficient use of memory.

* Flexibility: The API offers a range of tools for data transformation & augmentation, making it highly flexible. Users can easily apply complex transformations to their data, such as shuffling, batching, & augmenting datasets, which is crucial for robust ML model training.

* Easy Integration: It integrates seamlessly with TensorFlow's ML models & workflows. The API provides a convenient way to feed data into TensorFlow models, simplifying the code & reducing the risk of errors.

* Performance Optimization: The Dataset API allows for various optimizations like prefetching, where it preloads data into memory for faster access, & caching, where processed data can be kept in memory for quicker reuse. These optimizations significantly improve the efficiency of the training process.

* Consistency & Reusability: It provides a consistent & reusable interface for loading & preprocessing data. This consistency is beneficial for experimenting with different models or datasets, as it minimizes the need for code changes.

* Compatibility with Different Data Sources: The API can handle data from a wide variety of sources, including simple arrays, files, & even more complex data formats. This compatibility makes it versatile & useful for different types of ML projects.



In [ ]:
# @title
#@title Example of Data Sampling
'''
Runtime: GPU $$$
TensorFlow Datasets:       https://www.tensorflow.org/api_docs/python/tf/keras/datasets
CIFAR10:                   https://www.cs.toronto.edu/~kriz/cifar.htm
TensorFlow Data Pipelines: https://www.tensorflow.org/guide/data
'''

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.utils import to_categorical

# Load CIFAR-10 data
(datain_tr, dataou_tr), _ = cifar10.load_data()
datain_tr = datain_tr.astype('float32') / 255
dataou_tr = to_categorical(dataou_tr, 10)

# Create a TensorFlow Dataset
dataset = tf.data.Dataset.from_tensor_slices((datain_tr, dataou_tr))

# Sample a subset of the data using the Dataset API
num_samples = 5000  # Number of samples to use
dataset_sampled = dataset.shuffle(buffer_size=50000).take(num_samples).batch(64)

# Define a simple CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Fit the model on the sampled data
model.fit(dataset_sampled, epochs=10)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

## **7. High-Dimensional Data**

High-dimensional data presents significant challenges in ML, often described in the context of the “[curse of dimensionality](https://www.datacamp.com/blog/curse-of-dimensionality-machine-learning)”. This term refers to the various phenomena that arise when analyzing & organizing data in high-dimensional spaces that do not occur in lower-dimensional settings. As the number of features (dimensions) in a dataset increases, the volume of the space increases exponentially, making the available data sparse. This sparsity is problematic because predictive models have a harder time when they are not trained on dense data, leading to overfitting where the model learns noise in the training data rather than the actual underlying patterns. Additionally, high-dimensional data can significantly increase the computational complexity & memory requirements, making it difficult to process & analyze the data efficiently. This complexity also impacts the interpretability of the model, as it becomes more challenging to understand how specific features influence the model's predictions.

Here are a few examples with respect to Mechanical Engineering:

> **Advanced Manufacturing**: In advanced manufacturing, high-dimensional data can complicate the process of monitoring & quality control, where models might need to analyze a vast array of sensor data. The complexity of such data can lead to less accurate predictions & inefficient monitoring of manufacturing processes.

> **Biomechanical Engineering**: High-dimensional data in biomechanical engineering, such as data from motion capture systems or multiple sensor readings, can make it challenging to develop models that accurately predict or diagnose biomechanical issues, leading to less reliable outcomes in prosthetic design or rehabilitation strategies.

> **Fluid Mechanics & Thermal Science**: For fluid mechanics & thermal science, handling high-dimensional simulation data for complex fluid dynamics or thermal processes can be problematic, potentially leading to models that are computationally expensive & less accurate in predicting fluid or thermal behaviors.

> **Hypersonic Technologies**: In hypersonic technologies, dealing with high-dimensional data from various sensors & simulations can complicate the predictive modeling, making it challenging to accurately forecast the performance & behavior of hypersonic vehicles.

> **Robotics, Dynamics, & Controls**: Robotics involves processing high-dimensional data from sensors & control systems, which can lead to difficulties in developing efficient algorithms for real-time decision-making & control in dynamic environments.

> **Solids/Mechanics of Materials**: In the field of solids & mechanics of materials, analyzing high-dimensional data such as stress, strain, & thermal properties under various conditions can be challenging, potentially leading to less accurate material behavior predictions & hindering material development.


High-dimensional spaces can make learning difficult due to the curse of dimensionality. In this section, we will see a few strategies to address this issue.

### **7.1. Feature Selection**

[Feature selection](https://machinelearningmastery.com/feature-selection-with-real-and-categorical-data/) is a crucial technique in NNs & ML to address the challenges posed by high-dimensional data. High-dimensional datasets, which contain a large number of features, can lead to issues like overfitting, increased computational complexity, & difficulties in model interpretation. Feature selection involves identifying & selecting a subset of relevant features for use in model construction. This process helps in reducing the dimensionality of the data, mitigating the curse of dimensionality. By focusing on the most informative features, feature selection enhances the model's performance by eliminating irrelevant or redundant data that could mislead the learning process. It also reduces the computational cost & simplifies the model, making it easier to interpret. Effective feature selection can lead to more accurate, efficient, & interpretable models. In NNs, this can be particularly beneficial as it helps in optimizing the network architecture, reducing the number of input neurons, & thus the complexity of the model. In the context of high-dimensional data, feature selection is a vital step towards building models that are not only performant but also scalable & understandable.

In [ ]:
# @title
#@title Example of Feature Selection
'''
Runtime: CPU
make_classification: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
Feature selection:   https://scikit-learn.org/stable/modules/feature_selection.html
SelectKBest:         https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html
f_classif:           https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.f_classif.html
'''
import numpy as np
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectKBest, f_classif

# Generate synthetic high-dimensional classification dataset
random_state = np.random.randint(1000)
datain_tr, dataou_tr = make_classification(n_samples=1000, n_features=100, n_informative=10, random_state=random_state)
datain_vl, dataou_vl = make_classification(n_samples=200, n_features=100, n_informative=10, random_state=random_state)

# Feature selection using SelectKBest
selector = SelectKBest(f_classif, k=10)
datain_tr_selected = selector.fit_transform(datain_tr, dataou_tr)
datain_vl_selected = selector.transform(datain_vl)

# Define a simple TensorFlow model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(datain_tr_selected.shape[1],)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Fit the model
model.fit(datain_tr_selected, dataou_tr, epochs=10, validation_data=(datain_vl_selected, dataou_vl))

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **7.2. Feature Extraction**

[Feature extraction](https://deepai.org/machine-learning-glossary-and-terms/feature-extraction) is a fundamental technique in NNs & ML that addresses the complexities associated with high-dimensional data. Unlike feature selection, which involves choosing a subset of the original features, feature extraction is about transforming or combining the existing features to produce a new set of features. This process is crucial in managing high-dimensional datasets as it aims to reduce the number of features, thereby decreasing the dimensionality of the data. By doing so, feature extraction helps in alleviating the curse of dimensionality, improving model performance, & reducing computational costs. Techniques such as [Principal Component Analysis (PCA)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) in traditional ML or the use of convolutional layers in NNs are common methods of feature extraction. These methods transform the high-dimensional data into a lower-dimensional space where the most significant & informative aspects of the data are retained. This transformation not only simplifies the data & makes it more manageable but also often enhances the model's ability to learn & generalize from the data. In deep learning, feature extraction is inherently part of the process, as deep NNs automatically discover & learn the most relevant features from the data during training. Effective feature extraction is key to dealing with high-dimensional data, leading to more efficient, accurate, & robust ML models.

In [ ]:
# @title
#@title Example of Feature Extraction
'''
Runtime: CPU
make_classification: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html
PCA:              : https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
'''

import numpy as np
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA

# Generate synthetic high-dimensional classification dataset
random_state = np.random.randint(1000)
datain_tr, dataou_tr = make_classification(n_samples=1000, n_features=100, n_informative=10, random_state=random_state)
datain_vl, dataou_vl = make_classification(n_samples=200, n_features=100, n_informative=10, random_state=random_state)

# Feature extraction using PCA
pca = PCA(n_components=10, random_state=random_state)
datain_tr_pca = pca.fit_transform(datain_tr)
datain_vl_pca = pca.transform(datain_vl)

# Define a simple TensorFlow model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(datain_tr_pca.shape[1],)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Fit the model
model.fit(datain_tr_pca, dataou_tr, epochs=10, validation_data=(datain_vl_pca, dataou_vl))

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

# <font color="#418FDE" size="6.5" uppercase>**D: ML Major Challenges**</font>
----


In this lecture, you learned to
* Explain key challenges in ML, such as unbalanced data, overfitting, & gradient-related issues, & demonstrate how to address them.


In the next Module (Module 2), we will continue with the fundamentals recap.